# 09 · The whole pipeline end to end

Trains every model into a **fresh folder**, scores the human data, recomputes every number
and redraws Figures 1–4. Nothing already on disk is overwritten.

* **Synthetic data (default when the original images are absent):** the full pipeline,
  including the LR cross-validation, on the synthetic dataset; ≈ 45 min with a GPU.
  Finished runs from notebooks 02–05 are reused. The scores describe synthetic images only.
* **Original data:** re-trains the manuscript's models and prints a row-by-row comparison
  with the published values. Times on the reference machine:

| step | time on the reference machine (RTX, 8 GB; 14 GB RAM) |
|---|---|
| ResNet-50 ×2, MLP ×1, on 3 tasks | ≈ 1.5 h |
| LR feature cache + CV + refit | ≈ 12 h; with `REUSE_LR_CV = True` (reuse the reference CV table) ≈ 30 min |
| XGBoost ×2 on 3 tasks | ≈ 70 min |
| human transfer, numbers, figures | ≈ 10 min |

The classical models (LR, XGBoost) and the morphometric-only MLP are deterministic and
reproduce the manuscript's predictions exactly. ResNet-50 retraining is **not**
bit-reproducible, even on the original machine. Patches load on a thread pool, so the
random-flip order depends on thread timing. Each retrain is a new draw of the same
procedure, and its ResNet-50 rows differ from the manuscript, most at slide level.
Notebook 08 shows that the shipped ResNet-50 weights reproduce the published numbers
exactly.

From a terminal: `python reproduce.py --synthetic all` (synthetic) or
`python reproduce.py all` (original data).

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
# The histology images are not distributed. Without them this notebook runs on the
# synthetic dataset (scripts/make_synthetic_data.py) and writes to runs/synthetic/.
SYNTHETIC = (os.environ.get("VML_SYNTHETIC") == "1"          # forced, or no original data
             or not ((HERE / "data" / "Dataset_balanced" / "dataset.yaml").is_file() or "VML_DATA" in os.environ))
if SYNTHETIC:
    os.environ["VML_SYNTHETIC"] = "1"
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
if SYNTHETIC:   # generate and balance the synthetic data once (both steps skip existing output)
    subprocess.run([PY, str(HERE / "scripts" / "make_synthetic_data.py"), "--out", str(P.DATASETS)], check=True)
    subprocess.run([PY, str(HERE / "scripts" / "build_dataset.py"), "--only", "Dataset_balanced"],
                   check=True, capture_output=True)
print("mode           :", "SYNTHETIC data (not the manuscript's)" if SYNTHETIC else "original data")
print("dataset        :", P.DATA)
print("human data     :", P.HUMAN_ROOT)
print("results (used) :", P.RESULTS)

In [ ]:
GO = SYNTHETIC                   # <- True to run everything (original data: hours)
REUSE_LR_CV = not SYNTHETIC      # original data: reuse reference/lr_tuning_cv.csv instead of the ~12 h CV
BASE = P.WORK / "rerun"
R = P.roots(BASE)
print({k: str(v.relative_to(P.ROOT)) for k, v in R.items()})

In [ ]:
if GO:
    import lr_tuning, run_xgb
    # 1. ResNet-50, ResNet-50 + morph, MLP morph-only
    subprocess.run([PY, "-u", str(P.VML / "tools" / "train_eval_all.py"), "--data-root", str(P.DATA),
                    "--results-root", str(R["cnn"]), "--models", "cnn,cnn_morph,mlp_morph"], cwd=str(P.VML), check=True)
    # 2. LR: CV (or the reference table), then one refit per (task, model)
    if not REUSE_LR_CV:
        lr_tuning.extract(); lr_tuning.cv()
    lr_tuning.final(R["lr"], None if not REUSE_LR_CV else P.REFERENCE / "lr_tuning_cv.csv")
    # 3. XGBoost
    run_xgb.main(R["xgb"])
    # 4. Human transfer of the ResNet-50s (selftest first)
    for extra in (["--selftest"], []):
        subprocess.run([PY, "-u", str(HERE / "scripts" / "human_eval.py"), "--data-root", str(P.DATA),
                        "--results-root", str(R["cnn"]), "--models", "cnn,cnn_morph", *extra], check=True)
else:
    print("GO is False: nothing trained. Comparing whatever is already in", BASE)

In [ ]:
from manuscript_numbers import compute, compare
from manuscript_figures import summary_table
N = compute(BASE)
print(f"{len(N['rat'])} of 21 rat runs and {len(N['human'])} of 6 human runs found in {BASE.relative_to(P.ROOT)}")
if SYNTHETIC:
    result = summary_table(N, BASE).drop(columns="source", errors="ignore")   # synthetic: nothing to compare with
else:
    cmp = compare(N)
    print(f"{(cmp.status == 'match').sum()} of {len(cmp)} manuscript numbers reproduced to 3 decimals")
    result = cmp.assign(model=cmp.run.str.split("__").str[0]).groupby(["part", "model"]).status.value_counts().unstack(fill_value=0)
result

In [ ]:
if not SYNTHETIC:
    display(cmp[cmp.status == "DIFFERS"].assign(diff=lambda d: d["diff"].round(4)))

In [ ]:
if len(N["rat"]) == 21 and len(N["human"]) == 6:
    from manuscript_figures import make_figures
    from IPython.display import Image, display
    out = P.WORK / ("figures" if SYNTHETIC else "figures_rerun")
    make_figures(N, out, BASE)
    for name in ["figure1_patch_f1", "figure2_roc", "figure3_wsi_f1", "figure4_aclr_transfer"]:
        print(name); display(Image(filename=str(out / f"{name}.png"), width=800))